In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, dropout=0.2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=layer_dim,
            batch_first=True,
            dropout=dropout if layer_dim > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)

        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        return out

In [ ]:

def plot_loss(loss_list, test_loss_list):
    fig, ax1 = plt.subplots(figsize=(12, 10))

    # Loss
    ax1.plot(loss_list, label='Treino', linewidth=2)
    ax1.plot(test_loss_list, label='Teste', linewidth=2)
    ax1.set_xlabel("Época")
    ax1.set_ylabel("MSE Loss")
    ax1.set_title("Evolução da Loss durante o Treino")
    ax1.legend()
    ax1.grid(True)
    ax1.set_yscale('log')

    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.show()


def simulate_rnn_trajectory_xy(model, init_seq, scaler_xy, steps, noise_std=0.01):
    model.eval()
    current_seq = init_seq.clone().unsqueeze(0).to(device)
    trajectory = []

    with torch.no_grad():
        for step in range(steps):
            next_step = model(current_seq)

            """
            # Adicionar ruído controlado
            if noise_std > 0 and step > 0:  # Não adicionar no primeiro passo
                noise = torch.randn_like(next_step) * noise_std
                next_step = next_step + noise
            """

            trajectory.append(next_step.squeeze().cpu().numpy())

            next_step_exp = next_step.unsqueeze(1)
            current_seq = torch.cat([current_seq[:, 1:, :], next_step_exp], dim=1)

    trajectory = np.array(trajectory)
    trajectory = scaler_xy.inverse_transform(trajectory)
    x_pred, y_pred = trajectory[:, 0], trajectory[:, 1]
    return x_pred, y_pred


def find_corresponding_real_data(example_idx, datasets, idx_test, pos_test, window, steps):
    dataset_id = idx_test[example_idx].item()
    start_i = pos_test[example_idx].item()

    real_data = datasets[dataset_id]

    start_pos = start_i + window
    end_pos = min(start_pos + steps, len(real_data["x_obs"]))
    actual_steps = end_pos - start_pos

    x_real = real_data["x_obs"][start_pos:end_pos]
    y_real = real_data["y_obs"][start_pos:end_pos]

    return x_real, y_real, actual_steps


def plot_trajectory(datasets, featuresTest, model, idx_test, pos_test, scaler_xy, window, test_examples):
    min_steps = 15
    i = 0
    tested_examples = []

    for example_idx in test_examples:
        if example_idx >= len(featuresTest):
            continue

        print(f"\nExemplo: {example_idx} ---")

        init_seq = featuresTest[example_idx]
        desired_steps = 50

        x_pred, y_pred = simulate_rnn_trajectory_xy(model, init_seq, scaler_xy, steps=desired_steps, noise_std=0.005)

        x_real, y_real, actual_steps = find_corresponding_real_data(example_idx, datasets, idx_test, pos_test window, desired_steps)

        print(f"Predição: {len(x_pred)} pontos, Real: {len(x_real)} pontos")

        min_len = min(len(x_pred), len(x_real))
        x_pred = x_pred[:min_len]
        y_pred = y_pred[:min_len]
        x_real = x_real[:min_len]
        y_real = y_real[:min_len]

        position_error = np.sqrt((x_real - x_pred)**2 + (y_real - y_pred)**2)

        print(f"Exemplo {example_idx}:")
        print(f"Pontos comparados: {min_len}")
        print(f"Erro médio: {np.mean(position_error):.4f} m")
        print(f"Erro máximo: {np.max(position_error):.4f} m")

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

        ax1.plot(x_real, y_real, 'bo-', label="Real", markersize=4, alpha=0.7, linewidth=2)
        ax1.plot(x_pred, y_pred, 'rx-', label="Previsto (LSTM)", markersize=4, alpha=0.7, linewidth=2)
        ax1.set_xlabel("x [m]")
        ax1.set_ylabel("y [m]")
        ax1.set_title(f"Trajetória - Exemplo {example_idx}")
        ax1.legend()
        ax1.axis("equal")
        ax1.grid(True)


        ax2.plot(position_error, 'r-', linewidth=2)
        ax2.axhline(y=np.mean(position_error), color='b', linestyle='--',
                  label=f'Erro médio = {np.mean(position_error):.4f} m')
        ax2.set_xlabel('Passo de tempo')
        ax2.set_ylabel('Erro posicional (m)')
        ax2.set_title(f'Evolução do Erro')
        ax2.legend()
        ax2.grid(True)

        plt.tight_layout()
        plt.savefig(f'trajectory_comparison_{example_idx}.png')
        plt.show()


def plots(datasets, featuresTest, model, idx_test, window, test_examples, loss_list, test_loss_list):
    plot_loss(loss_list, test_loss_list)
    plot_trajectory(datasets, featuresTest, model, idx_test, window, test_examples)
    print("\n" + "="*50)
    print("ESTATÍSTICAS FINAIS")
    print("="*50)
    print(f"Épocas treinadas: {len(loss_list)}")
    print(f"Final loss de treino: {loss_list[-1]:.6f}")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# carregando datasets
datasets = np.load("pendulum_datasets_with_noise.npy", allow_pickle=True)
print("Loaded:", len(datasets), "examples")

# janela de tempo pra previsão
window = 15

features_list = []
targets_list = []
dataset_idx_list = []

all_xy = np.concatenate([np.stack([d["x_obs"], d["y_obs"]], axis=-1) for d in datasets], axis=0)
scaler_xy = StandardScaler().fit(all_xy)

#cria sequencias com base na janela de tempo
for idx, data in enumerate(datasets):

    if len(data["x_obs"]) < window + 1:
        continue

    xy = np.stack([data["x_obs"], data["y_obs"]], axis=-1).astype(np.float32)
    xy_scaled = scaler_xy.transform(xy)

    Ns = len(xy_scaled)

    for i in range(Ns - window - 1):
        seq_in = xy_scaled[i:i+window]
        next_step = xy_scaled[i+window]

        features_list.append(seq_in)
        targets_list.append(next_step)
        dataset_idx_list.append(idx)


features_numpy = np.array(features_list, dtype=np.float32)
targets_numpy = np.array(targets_list, dtype=np.float32)
dataset_idx_numpy = np.array(dataset_idx_list)

print("features_numpy shape:", features_numpy.shape)
print("targets_numpy shape:", targets_numpy.shape)

featuresTensor = torch.tensor(features_numpy, dtype=torch.float32)
targetsTensor  = torch.tensor(targets_numpy, dtype=torch.float32)
dataset_idx_tensor = torch.tensor(dataset_idx_numpy)

# split 80/20
split = int(0.8 * len(featuresTensor))
featuresTrain, featuresTest = featuresTensor[:split], featuresTensor[split:]
targetsTrain, targetsTest = targetsTensor[:split], targetsTensor[split:]
idx_train, idx_test = dataset_idx_tensor[:split], dataset_idx_tensor[split:]

test_examples = [0, 55, 105]

Using device: cuda
Loaded: 500 examples
features_numpy shape: (4500, 15, 2)
targets_numpy shape: (4500, 2)


In [ ]:
batch_size = 256
num_epochs = 200

train_dataset = TensorDataset(featuresTrain, targetsTrain)
test_dataset  = TensorDataset(featuresTest, targetsTest)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Train batches:", len(train_loader))
print("Test batches :", len(test_loader))

input_dim = 2
hidden_dim = 128
layer_dim = 2
output_dim = 2

model = LSTMModel(input_dim, hidden_dim, layer_dim, output_dim).to(device)

error = nn.MSELoss()
learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)


Train batches: 15
Test batches : 4


In [ ]:
loss_list = []
test_loss_list = []
learning_rates = []

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = error(outputs, targets)
        loss.backward()
        #torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)

    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(device), y_test.to(device)
            y_pred = model(x_test)
            test_loss += error(y_pred, y_test).item()

    avg_test_loss = test_loss / len(test_loader)

    loss_list.append(avg_train_loss)
    test_loss_list.append(avg_test_loss)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        #print(f"Epoch {epoch+1}/{num_epochs} | Train: {avg_train_loss:.6f} | Test: {avg_test_loss:.6f} | LR: {current_lr:.6f}")
        print(f"Epoch {epoch+1}/{num_epochs} | Train: {avg_train_loss:.6f} | Test: {avg_test_loss:.6f}")

plots(datasets, featuresTest, model, idx_test, window, test_examples, loss_list, test_loss_list)

Epoch 1/200 | Train: 0.781811 | Test: 0.565745
Epoch 10/200 | Train: 0.093907 | Test: 0.074043
Epoch 20/200 | Train: 0.045529 | Test: 0.037102
Epoch 30/200 | Train: 0.032274 | Test: 0.026731
Epoch 40/200 | Train: 0.028752 | Test: 0.026140
Epoch 50/200 | Train: 0.029901 | Test: 0.024711
Epoch 60/200 | Train: 0.026099 | Test: 0.023246
Epoch 70/200 | Train: 0.026425 | Test: 0.022248
Epoch 80/200 | Train: 0.025754 | Test: 0.022178
Epoch 90/200 | Train: 0.024196 | Test: 0.022362
Epoch 100/200 | Train: 0.025239 | Test: 0.023430
Epoch 110/200 | Train: 0.024689 | Test: 0.022743


KeyboardInterrupt: 

In [ ]:
def physical_loss(inputs, delta_t = 0.1, L = 1.0):
        x_t_minus_1 = inputs[:, -2, 0]
        y_t_minus_1 = inputs[:, -2, 1]
        x_t = inputs[:, -1, 0]
        y_t = inputs[:, -1, 1]

        acc_x_pred = (x_pred - 2 * x_t + x_t_minus_1) / (delta_t ** 2)
        acc_y_pred = (y_pred - 2 * y_t + y_t_minus_1) / (delta_t ** 2)

        mean_xy = torch.tensor(scaler_xy.mean_, dtype=torch.float32).to(device)
        std_xy = torch.tensor(scaler_xy.scale_, dtype=torch.float32).to(device)

        x_t_unscaled = x_t * std_xy[0] + mean_xy[0]
        y_t_unscaled = y_t * std_xy[1] + mean_xy[1]


        theta_t = torch.atan2(x_t_unscaled, -y_t_unscaled)
        acc_x_phys = -(9.81 / L) * torch.sin(theta_t)
        acc_y_phys = -(9.81 / L) * torch.cos(theta_t)

        acc_x_phys_scaled = acc_x_phys / (std_xy[0] / delta_t**2)
        acc_y_phys_scaled = acc_y_phys / (std_xy[1] / delta_t**2)


        acc_x_pred_unscaled = acc_x_pred * std_xy[0]
        acc_y_pred_unscaled = acc_y_pred * std_xy[1]

        return torch.mean((acc_x_pred_unscaled - acc_x_phys) ** 2 + (acc_y_pred_unscaled - acc_y_phys) ** 2)

In [ ]:
loss_list = []
test_loss_list = []
learning_rates = []

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)

        x_pred, y_pred = outputs[:, 0], outputs[:, 1]
        x_real, y_real = targets[:, 0], targets[:, 1]

        loss_data = error(x_pred, x_real) + error(y_pred, y_real)

        loss_phys = physical_loss(inputs)

        loss = loss_data + 1e-5 * loss_phys
        loss.backward()
        #torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)

    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(device), y_test.to(device)
            y_pred = model(x_test)
            test_loss += error(y_pred, y_test).item()

    avg_test_loss = test_loss / len(test_loader)

    loss_list.append(avg_train_loss)
    test_loss_list.append(avg_test_loss)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        #print(f"Epoch {epoch+1}/{num_epochs} | Train: {avg_train_loss:.6f} | Test: {avg_test_loss:.6f} | LR: {current_lr:.6f}")
        print(f"Epoch {epoch+1}/{num_epochs} | Train: {avg_train_loss:.6f} | Test: {avg_test_loss:.6f}")

plots(datasets, featuresTest, model, idx_test, window, test_examples, loss_list, test_loss_list)

In [9]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image

# --- Configurações iniciais (mantidas) ---
os.makedirs("plots_lstm", exist_ok=True) # Alterando a pasta de plots
np.random.seed(123)
torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def save_gif_PIL(outfile, files, fps=15, loop=0):
    imgs = [Image.open(f) for f in files]
    imgs[0].save(outfile, save_all=True, append_images=imgs[1:],
                 duration=int(1000/fps), loop=loop)

# --- Modelo de Rede Neural (Substituição da FCN) ---

# Adaptação para usar LSTM
# Mapeamento t -> theta
class LSTM_Model(nn.Module):
    def __init__(self, N_INPUT, N_OUTPUT, N_HIDDEN, N_LAYERS):
        super().__init__()

        # LSTM: input_size=N_INPUT (1, que é o tempo), hidden_size=N_HIDDEN, num_layers=N_LAYERS
        self.lstm = nn.LSTM(
            input_size=N_INPUT,
            hidden_size=N_HIDDEN,
            num_layers=N_LAYERS,
            batch_first=True # O formato será (batch, seq, feature)
        )

        # Camada Linear para mapear o hidden state final para a saída (theta)
        self.linear = nn.Linear(N_HIDDEN, N_OUTPUT)

    def forward(self, x):
        # x tem formato (batch, 1) [Ex: (10, 1) para dados de treinamento]

        # 1. Adicionar dimensão de sequência (seq_len=1): (batch, 1, feature)
        x_reshaped = x.unsqueeze(1)

        # 2. Passar pela LSTM
        # out: (batch, seq, hidden_size). h_n, c_n: (num_layers, batch, hidden_size)
        lstm_out, (h_n, c_n) = self.lstm(x_reshaped)

        # 3. Pegar a saída do último passo da sequência (índice -1) ou o último hidden state (h_n[-1])
        # Usaremos o hidden state da última camada (h_n[-1])
        # h_n[-1] tem formato (batch, hidden_size)

        # 4. Passar pela camada linear para obter a saída final (theta)
        out = self.linear(h_n[-1])
        return out

# --- Carregamento e Preparação dos Dados (Mantidos) ---

all_data = np.load("pendulum_datasets_with_noise.npy", allow_pickle=True)
sample = all_data[6]

params = sample["params"]
print("Loaded parameters:", params)

t_dense = sample["t_dense"]
theta_dense = sample["theta_dense"]
# omega_dense = sample["omega_dense"]
# x_dense = sample["x_dense"]
# y_dense = sample["y_dense"]

t_obs = sample["t_obs"]
x_obs = sample["x_obs"]
y_obs = sample["y_obs"]

# choose number of datapoints, try extra polation
N = 10
t_obs = t_obs[:N]
x_obs = x_obs[:N]
y_obs = y_obs[:N]

g = float(params["g"])
L = float(params["L"])
b = float(params["b"])

# NORMALIZATION (Mantido)
t_min, t_max = t_dense.min(), t_dense.max()
print(f"t_min, t_max : {t_min}, {t_max}")

def norm_t(t):
    return 2*(t - t_min)/(t_max - t_min) - 1

def denorm_t(tn):
    return (tn + 1)/2 * (t_max - t_min) + t_min

# scaling factor for derivatives (Mantido)
scale = 2.0 / (t_max - t_min)
print(f"scale: {scale}")

t_dense_n = norm_t(t_dense)
t_obs_n   = norm_t(t_obs)

# convert to torch
t_dense_t = torch.tensor(t_dense_n, dtype=torch.float32, device=device).view(-1,1)
t_obs_t   = torch.tensor(t_obs_n,   dtype=torch.float32, device=device).view(-1,1)

x_obs_t = torch.tensor(x_obs, dtype=torch.float32, device=device).view(-1,1)
y_obs_t = torch.tensor(y_obs, dtype=torch.float32, device=device).view(-1,1)

theta_dense_t = torch.tensor(theta_dense, dtype=torch.float32, device=device).view(-1,1)

theta_obs = np.arctan2(x_obs, -y_obs)


# Plot (Mantido, apenas alteração no diretório de salvamento)

def plot_result(t_dense, theta_true, t_obs_n, theta_data, theta_pred, step, title, fname):
    plt.figure(figsize=(8,4))
    plt.plot(t_dense, theta_true, color="tab:green", linewidth=2, alpha=0.8, label="Exact $\\theta(t)$")

    # pred
    plt.plot(t_dense, theta_pred, color="tab:blue", linewidth=3, alpha=0.85, label="Network prediction")

    t_obs_denorm = denorm_t(t_obs_n)
    plt.scatter(norm_t(t_obs_denorm), theta_data, s=60, color="tab:orange",
                alpha=0.7, label="Training data")

    plt.text(1.05, 0.7, f"step: {step}", transform=plt.gca().transAxes)
    l = plt.legend(loc=(1.01,0.15), frameon=False)
    plt.setp(l.get_texts(), color="k")
    plt.xlim(-1.05,1.05)
    plt.ylim(-2,2)
    plt.axis("off")
    plt.savefig(fname, dpi=120, bbox_inches='tight', pad_inches=0.1)
    plt.close()


# --- NN com LSTM (sem física) ---

print("\nTraining NN with LSTM...")
# Usamos a nova classe LSTM_Model
model_lstm_nn = LSTM_Model(1, 1, 32, 3).to(device)
opt_lstm_nn = torch.optim.Adam(model_lstm_nn.parameters(), lr=1e-3)

files_nn_lstm = []

"""for i in range(10000):
    opt_lstm_nn.zero_grad()
    theta_pred = model_lstm_nn(t_obs_t)

    x_pred = L * torch.sin(theta_pred)
    y_pred = -L * torch.cos(theta_pred)

    loss = torch.mean((x_pred - x_obs_t)**2 + (y_pred - y_obs_t)**2)
    loss.backward()
    opt_lstm_nn.step()

    if (i+1) % 10 == 0 or i == 0:
        with torch.no_grad():
            th_full = model_lstm_nn(t_dense_t).cpu().numpy().flatten()

        fname = f"plots_lstm/lstm_nn_{i+1:05d}.png"
        plot_result(
            t_dense_n, theta_dense, t_obs_n, theta_obs, th_full,
            i+1, "LSTM NN", fname
        )
        files_nn_lstm.append(fname)

save_gif_PIL("lstm_nn.gif", files_nn_lstm, fps=15)
print("Saved lstm_nn.gif")
"""

# --- PINN com LSTM ---

print("\nTraining PINN with LSTM...")
model_lstm_pinn = LSTM_Model(1, 1, 32, 3).to(device)
opt_lstm_pinn = torch.optim.Adam(model_lstm_pinn.parameters(), lr=1e-4)

t_col = np.linspace(t_dense.min(), t_dense.max(), 200)
t_col_n = torch.tensor(norm_t(t_col), dtype=torch.float32, device=device).view(-1,1)
t_col_n.requires_grad_(True)

files_pinn_lstm = []

for i in range(20000):
    opt_lstm_pinn.zero_grad()

    # data loss
    theta_d = model_lstm_pinn(t_obs_t)
    x_pred = L * torch.sin(theta_d)
    y_pred = -L * torch.cos(theta_d)
    loss_data = torch.mean((x_pred - x_obs_t)**2 + (y_pred - y_obs_t)**2)

    # physics loss
    with torch.backends.cudnn.flags(enabled=False):
        # physics loss
        t_col_n.requires_grad_(True) # Garante que o gradiente está ativo

        theta_col = model_lstm_pinn(t_col_n)

        # 1ª Derivada: d(theta)/dt_n
        dtheta_dt_n = torch.autograd.grad(theta_col, t_col_n,
                                         torch.ones_like(theta_col),
                                         create_graph=True)[0]

        # 2ª Derivada: d2(theta)/dt_n2 (Este é o passo que falhava)
        d2theta_dt2_n = torch.autograd.grad(dtheta_dt_n, t_col_n,
                                             torch.ones_like(dtheta_dt_n),
                                             create_graph=True)[0]

        # scale derivatives
        dtheta_dt = scale * dtheta_dt_n
        d2theta_dt2 = scale**2 * d2theta_dt2_n

        # physics ODE
        phys = d2theta_dt2 + (b/L) * dtheta_dt + (g/L) * torch.sin(theta_col)

        loss_phys = torch.mean(phys**2)

    # weighted loss
    loss = loss_data + 0.01 * loss_phys
    loss.backward()
    opt_lstm_pinn.step()

    if (i+1) % 10 == 0 or i == 0:
        with torch.no_grad():
            th_full = model_lstm_pinn(t_dense_t).cpu().numpy().flatten()

        fname = f"plots_lstm/lstm_pinn_{i+1:05d}.png"
        plot_result(
            t_dense_n, theta_dense, t_obs_n, theta_obs, th_full,
            i+1, "LSTM PINN", fname
        )
        files_pinn_lstm.append(fname)

save_gif_PIL("lstm_pinn.gif", files_pinn_lstm, fps=15)
print("Saved lstm_pinn.gif")
print("Done.")

Device: cuda
Loaded parameters: {'g': 9.81, 'L': 1.2, 'b': 0.1, 'theta0': 0.5, 'omega0': 0.0, 'noise_sigma': 0.01}
t_min, t_max : 0.0, 9.999999999999831
scale: 0.20000000000000337

Training NN with LSTM...

Training PINN with LSTM...
Saved lstm_pinn.gif
Done.
